# Backbone Comparison Experiments

Which backbone architecture provides the best mAP-efficiency tradeoff for jaguar re-identification?

**Tested Backbones:**
- **DINOv3-Large** (vit_large_patch16_dinov3.lvd1689m, 1024-dim, latest self-supervised ViT, best generalization)
- **DINOv3-Base** (vit_base_patch16_dinov3.lvd1689m, 768-dim, efficient DINOv3 variant)
- **DINOv2-Large** (1024-dim, previous-generation self-supervised ViT)
- **DINOv2-Base** (768-dim, balanced)
- **DINOv2-Small** (384-dim, fast)
- **MegaDescriptor-L-384** (1536-dim, trained on wildlife re-ID datasets)
- **MegaDescriptor-B-224** (768-dim, animal re-ID specialist)
- **ResNet50** (2048-dim, CNN baseline)
- **ConvNeXt Base** (1024-dim, modern CNN)
- **ConvNeXtV2 Base** (1024-dim, modern CNN v2 with MAE pretraining)
- **EfficientNet B3** (1536-dim, efficient CNN)

**Fixed Settings:**
- Loss: ArcFace (m=0.5, s=64)
- Dataset: JID_Master_Dataset (segmented, deduplicated)
- Epochs: 50

Results logged to Wandb project: `camera-trap-reidentification`, group: `backbone_comparison`

## Setup

In [1]:
import sys
from pathlib import Path
import logging

# Add src to path
project_root = Path.cwd().parent / "camera-trap-footage"
sys.path.insert(0, str(project_root / "src"))

from jaguars.common.logging_utils import setup_logger
from jaguars.reidentification.config import get_default_config
from jaguars.reidentification.experiments import get_backbone_experiments
from jaguars.reidentification.training.train import run_processing as run_training

logger = setup_logger("backbone_experiments", level=logging.INFO)
print("✓ Imports successful")

✓ Imports successful


## Configuration

Configure base settings for all backbone experiments.

In [2]:
# Get default configuration
config = get_default_config()

# Wandb settings
config.wandb.enabled = True
config.wandb.entity = "jaguars"
config.wandb.project = "camera-trap-reidentification"
config.wandb.tags = ["backbone_experiment", "arcface_hard_loss"]

# Dataset settings
config.dataset.source = "fiftyone"
config.dataset.fo_dataset_name = "JID_Master_Dataset"
config.dataset.fo_split_field = "closed_set_split"
config.dataset.fo_patches_field = "sam3_segmentations"
config.dataset.fo_label_field = "ground_truth"

# Training settings
config.training.num_epochs = 50
config.training.loss_name = "arcface"
config.model.arcface_margin = 0.7
config.model.arcface_scale = 64.0

print(f"✓ Base config loaded")
print(f"  Wandb project: {config.wandb.project}")
print(f"  Dataset: {config.dataset.fo_dataset_name}")
print(f"  Loss: {config.training.loss_name}")
print(f"  Epochs: {config.training.num_epochs}")

✓ Base config loaded
  Wandb project: camera-trap-reidentification
  Dataset: JID_Master_Dataset
  Loss: arcface
  Epochs: 50


## Load Experiments

In [3]:
# Get backbone experiments with our custom config
backbone_experiments = get_backbone_experiments(base_config=config)

print(f"✓ {len(backbone_experiments)} backbone experiments configured:")
for exp in backbone_experiments:
    print(f"  - {exp.name}: {exp.description}")

✓ 9 backbone experiments configured:
  - backbone_vit_large_patch14_dinov2.lvd142m: DINOv2 Large (1024-dim, best quality)
  - backbone_vit_base_patch14_dinov2.lvd142m: DINOv2 Base (768-dim, balanced)
  - backbone_vit_small_patch14_dinov2.lvd142m: DINOv2 Small (384-dim, fast)
  - backbone_hf-hub:BVRA_MegaDescriptor-L-384: MegaDescriptor Large 384 (animal re-ID specialist)
  - backbone_hf-hub:BVRA_MegaDescriptor-B-224: MegaDescriptor Base 224 (animal re-ID specialist)
  - backbone_resnet50: ResNet50 (25M params, CNN baseline)
  - backbone_convnext_base: ConvNeXt Base (modern CNN)
  - backbone_convnextv2_base.fcmae_ft_in22k_in1k: ConvNeXtV2 Base (88M params, modern CNN v2)
  - backbone_efficientnet_b3: EfficientNet B3 (efficient CNN)


## Run Experiments

Train each backbone with fixed ArcFace loss and compare results.

In [ ]:
# Run all backbone experiments
results = {}

for experiment in backbone_experiments:
    logger.info(f"Running experiment: {experiment.name}")
    logger.info(f"  Description: {experiment.description}")
    logger.info(f"  Backbone: {experiment.base_config.backbone.name}")
    logger.info(f"  Embedding dim: {experiment.base_config.backbone.embedding_dim}")
    logger.info(f"  Tags: {experiment.base_config.wandb.tags}")
    logger.info(f"  Group: {experiment.group}")
    
    try:
        # Run training
        result = run_training(experiment.base_config)
        results[experiment.name] = result
        logger.info(f"✓ {experiment.name} completed")
    except Exception as e:
        logger.error(f"✗ {experiment.name} failed: {e}")
        results[experiment.name] = {"error": str(e)}

print(f"\n✓ All {len(backbone_experiments)} backbone experiments completed")

04:35:48 - jid_logger.backbone_experiments - INFO - Running experiment: backbone_vit_large_patch14_dinov2.lvd142m
04:35:48 - jid_logger.backbone_experiments - INFO -   Description: DINOv2 Large (1024-dim, best quality)
04:35:48 - jid_logger.backbone_experiments - INFO -   Backbone: vit_large_patch14_dinov2.lvd142m
04:35:48 - jid_logger.backbone_experiments - INFO -   Embedding dim: 1024
04:35:48 - jid_logger.backbone_experiments - INFO -   Tags: ['backbone_experiment', 'arcface_hard_loss']
04:35:48 - jid_logger.backbone_experiments - INFO -   Group: backbone_comparison
04:35:48 - jid_logger.reidentification.training - INFO - Starting re-identification training...
04:35:48 - jid_logger.reidentification.training - INFO - Dataset source: fiftyone
04:35:48 - jid_logger.reidentification.training - INFO - Backbone: vit_large_patch14_dinov2.lvd142m
04:35:48 - jid_logger.reidentification.training - INFO - Device: cuda


04:35:50 - jid_logger.reidentification.training - INFO - Resource validation passed


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /sc/home/philipp.kolbe/.netrc.
wandb: Currently logged in as: hpi-philipp-kolbe to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


04:35:52 - jid_logger.reidentification.training - INFO - Loading dataset...
04:36:05 - jid_logger.reidentification.training - INFO - Dataset loaded:
04:36:05 - jid_logger.reidentification.training - INFO -   Train: 946 samples
04:36:05 - jid_logger.reidentification.training - INFO -   Val: 120 samples
04:36:05 - jid_logger.reidentification.training - INFO -   Num classes: 76
04:36:05 - jid_logger.reidentification.training - INFO - Using pre-computed embeddings
04:36:05 - jid_logger.reidentification.training - INFO - DataLoaders created:
04:36:05 - jid_logger.reidentification.training - INFO -   Train batches: 30
04:36:05 - jid_logger.reidentification.training - INFO -   Val batches: 4
Model initialized:
  Input dim: 1536
  Hidden dim: 512
  Embedding dim: 256
  Num classes: 76
  ArcFace margin: 0.7
  ArcFace scale: 64.0
  Total parameters: 939,264
04:36:05 - jid_logger.reidentification.training - INFO - Loss: arcface
04:36:05 - jid_logger.reidentification.training - INFO - Training com

04:36:06 - jid_logger.reidentification.training - INFO - Train Loss: 50.3684, Train Acc: 0.00%
04:36:06 - jid_logger.reidentification.training - INFO - Val Loss: 48.8669, Val Acc: 0.00%
04:36:06 - jid_logger.reidentification.training - INFO - Val mAP (identity-balanced): 0.2655
04:36:06 - jid_logger.reidentification.training - INFO - Val CMC@1: 0.4078, CMC@5: 0.6893
04:36:06 - jid_logger.reidentification.training - INFO - Learning Rate: 0.000100


...

04:47:45 - jid_logger.reidentification.training - INFO - Saved best model to data/models/reidentification/best_model.pt
04:47:45 - jid_logger.reidentification.training - INFO - Saved checkpoint to data/models/reidentification/checkpoint_epoch_50.pt
04:47:45 - jid_logger.reidentification.training - INFO - ======================================================================
04:47:45 - jid_logger.reidentification.training - INFO - Training completed!
04:47:45 - jid_logger.reidentification.training - INFO - Best epoch: 50
04:47:45 - jid_logger.reidentification.training - INFO - Best val_map: 0.6821


epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/acc,▁▁▁▁▁▁▁▁▂▂▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▇▇▇▇████
train/batch_acc,▁▁▁▁▁▁▁▁▁▂▃▃▃▄▃▃▄▃▄▄▄▅▅▅▅▆▅▅▆▅▇▆▆▅▇▅█▇▇▇
train/batch_cls_loss,█▇▇▆▆▆▆▆▅▅▅▄▅▅▅▄▄▄▃▃▄▄▃▃▅▃▄▃▁▂▃▂▂▃▂▂▂▂▁▂
train/batch_loss,██▇▆▆▆▅▆▄▄▄▄▄▄▅▅▃▄▅▃▃▄▄▃▃▃▃▂▃▃▂▃▃▂▂▂▃▁▃▂
train/loss,█▇▇▇▇▆▆▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁
val/acc,▁▁▁▁▁▁▂▂▃▃▃▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇█▇█████
val/cmc@1,▁▁▁▁▁▃▃▃▃▃▄▄▄▄▆▆▆▆▆▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇████
val/cmc@10,▁▁▂▂▂▄▄▄▄▄▅▅▅▅▆▅▆▇▇▆▇▇▇▇▇▇▇▇▇▇██▇███████
+12,...


04:49:09 - jid_logger.backbone_experiments - INFO - ✓ backbone_hf-hub:BVRA_MegaDescriptor-B-224 completed
04:49:09 - jid_logger.backbone_experiments - INFO - Running experiment: backbone_resnet50
04:49:09 - jid_logger.backbone_experiments - INFO -   Description: ResNet50 (25M params, CNN baseline)
04:49:09 - jid_logger.backbone_experiments - INFO -   Backbone: resnet50
04:49:09 - jid_logger.backbone_experiments - INFO -   Embedding dim: 2048
04:49:09 - jid_logger.backbone_experiments - INFO -   Tags: ['backbone_experiment', 'arcface_hard_loss']
04:49:09 - jid_logger.backbone_experiments - INFO -   Group: backbone_comparison
04:49:09 - jid_logger.reidentification.training - INFO - Starting re-identification training...
04:49:09 - jid_logger.reidentification.training - INFO - Dataset source: fiftyone
04:49:09 - jid_logger.reidentification.training - INFO - Backbone: resnet50
04:49:09 - jid_logger.reidentification.training - INFO - Device: cuda
04:49:09 - jid_logger.reidentification.train

04:49:11 - jid_logger.reidentification.training - INFO - Loading dataset...
04:49:23 - jid_logger.reidentification.training - INFO - Dataset loaded:
04:49:24 - jid_logger.reidentification.training - INFO -   Train: 946 samples
04:49:24 - jid_logger.reidentification.training - INFO -   Val: 120 samples
04:49:24 - jid_logger.reidentification.training - INFO -   Num classes: 76
04:49:24 - jid_logger.reidentification.training - INFO - Extracting embeddings with backbone...
Loading resnet50 model...
Model loaded successfully
  Parameters: 23,508,032
  Embedding dimension: 2048


Train embeddings:  33%|███▎      | 10/30 [00:13<00:16,  1.20it/s]

## Results Summary

Compare backbone performance on validation set.

In [ ]:
# Print summary of results
import pandas as pd

summary_data = []
for exp_name, result in results.items():
    if "error" in result:
        summary_data.append({
            "Experiment": exp_name,
            "Validation mAP": "ERROR",
            "Status": result["error"]
        })
    else:
        # Try different possible keys for mAP
        val_map = result.get("final_val_map") or result.get("best_val_map") or result.get("validation/map") or "N/A"
        summary_data.append({
            "Experiment": exp_name,
            "Validation mAP": f"{val_map:.4f}" if isinstance(val_map, (int, float)) else val_map,
            "Status": "✓ Completed"
        })

summary_df = pd.DataFrame(summary_data)
print("\n=== Backbone Comparison Results ===\n")
print(summary_df.to_string(index=False))
print(f"\nView detailed results at: https://wandb.ai/{config.wandb.entity}/{config.wandb.project}")


=== Backbone Comparison Results ===

                               Experiment Validation mAP                  Status
backbone_vit_large_patch14_dinov2.lvd142m          ERROR list index out of range
 backbone_vit_base_patch14_dinov2.lvd142m          ERROR list index out of range
backbone_vit_small_patch14_dinov2.lvd142m          ERROR list index out of range
                        backbone_resnet50          ERROR list index out of range
                   backbone_convnext_base          ERROR list index out of range
                 backbone_efficientnet_b3          ERROR list index out of range

View detailed results at: https://wandb.ai/jaguars/camera-trap-reidentification
